In [2]:
# ================================
# INSTALL (Colab Safe)
# ================================
!pip -q install plotly scikit-learn statsmodels

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing

np.random.seed(42)

# ================================
# 1) SYNTHETIC CAMPUS DATA
# ================================

days = 365
dates = pd.date_range("2025-01-01", periods=days)

df = pd.DataFrame({
    "date": dates
})

# synthetic signals
trend = np.linspace(0, 20, days)
season = 15 * np.sin(np.arange(days) * 2*np.pi/7)
noise = np.random.normal(0, 5, days)

df["energy_kwh"] = 300 + trend + season + noise
df["water_liters"] = 500 + trend*1.5 + noise*2
df["waste_kg"] = 120 + season + noise
df["solar_generation"] = 80 + 10*np.sin(np.arange(days)*2*np.pi/30)

num_cols = ["energy_kwh","water_liters","waste_kg","solar_generation"]
df[num_cols] = df[num_cols].clip(lower=1)

# ================================
# 2) ENSEMBLE MODEL
# (Regression + Exponential Smoothing)
# ================================

df["t"] = np.arange(len(df))

X = df[["t"]]
y = df["energy_kwh"]

lr = LinearRegression()
lr.fit(X, y)

df["lr_pred"] = lr.predict(X)

smooth_model = ExponentialSmoothing(
    df["energy_kwh"],
    trend="add",
    seasonal="add",
    seasonal_periods=7
).fit()

df["smooth_pred"] = smooth_model.fittedvalues

# Ensemble average
df["ensemble"] = (df["lr_pred"] + df["smooth_pred"]) / 2

# ================================
# 3) FUTURE FORECAST
# ================================

future_days = 60
future_index = np.arange(len(df), len(df)+future_days)

future_lr = lr.predict(future_index.reshape(-1,1))
future_smooth = smooth_model.forecast(future_days)

future_ensemble = (future_lr + future_smooth) / 2

future_dates = pd.date_range(df["date"].iloc[-1] + pd.Timedelta(days=1),
                             periods=future_days)

# ================================
# 4) KPI CALCULATIONS
# ================================

carbon_factor = 0.82   # kg CO2 per kWh
baseline = 350         # baseline campus energy

total_energy_saved = np.sum(baseline - df["energy_kwh"])
carbon_saved = total_energy_saved * carbon_factor
avg_water = df["water_liters"].mean()
avg_waste = df["waste_kg"].mean()

# ================================
# 5) DASHBOARD
# ================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Energy Usage + Ensemble Forecast",
        "Water Consumption",
        "Waste Generation",
        "Solar Generation"
    )
)

# ENERGY
fig.add_trace(
    go.Scatter(x=df["date"], y=df["energy_kwh"],
               name="Actual Energy"),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df["date"], y=df["ensemble"],
               name="Ensemble Fit",
               line=dict(dash="dash")),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=future_dates, y=future_ensemble,
               name="Forecast",
               line=dict(color="red", width=3)),
    row=1, col=1
)

# WATER
fig.add_trace(
    go.Scatter(x=df["date"], y=df["water_liters"],
               name="Water"),
    row=1, col=2
)

# WASTE
fig.add_trace(
    go.Bar(x=df["date"], y=df["waste_kg"],
           name="Waste"),
    row=2, col=1
)

# SOLAR
fig.add_trace(
    go.Scatter(x=df["date"], y=df["solar_generation"],
               name="Solar"),
    row=2, col=2
)

fig.update_layout(
    height=800,
    title="Campus Sustainability Tracker Dashboard",
    template="plotly_white"
)

fig.show()

# ================================
# 6) KPI PRINT
# ================================

print("\n===== CAMPUS SUSTAINABILITY KPIs =====")
print(f"Total Energy Saved: {total_energy_saved:.2f} kWh")
print(f"Carbon Emission Reduction: {carbon_saved:.2f} kg CO2")
print(f"Average Daily Water Use: {avg_water:.2f} liters")
print(f"Average Daily Waste: {avg_waste:.2f} kg")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(



===== CAMPUS SUSTAINABILITY KPIs =====
Total Energy Saved: 14581.85 kWh
Carbon Emission Reduction: 11957.12 kg CO2
Average Daily Water Use: 515.10 liters
Average Daily Waste: 120.05 kg
